# Lab 3: Experimentación de hiperparámetros.



El objetivo de este laboratorio es experimentar con los conceptos teoricos vistos en clase. Se propone seguir la estructura de experimentos del documento. Como hemos visto durante el tema es muy importante vuestra conclusión después del experimento.

Para evaluar con cual nos quedamos después de cada experimento vamos a quedarnos con el que tenga mejor Accuracy en los datos de validación.

El dataset a utilizar consiste en imágenes de personajes de los Simpsons extraídas directamente de capítulos de la serie. Este dataset ha sido recopilado por [Alexandre Attia](http://www.alexattia.fr/) y es más complejo que el dataset de Fashion MNIST que hemos utilizado hasta ahora. Aparte de tener más clases (vamos a utilizar los 18 personajes con más imágenes), los personajes pueden aparecer en distintas poses, en distintas posiciones de la imagen o con otros personajes en pantalla (si bien el personaje a clasificar siempre aparece en la posición predominante).

El dataset de training puede ser descargado desde aquí:

[Training data](https://onedrive.live.com/download?cid=C506CF0A4F373B0F&resid=C506CF0A4F373B0F%219337&authkey=AMzI92bJPx8Sd60) (~500MB)

Por otro lado, el dataset de test puede ser descargado de aquí:

[Test data](https://onedrive.live.com/download?cid=C506CF0A4F373B0F&resid=C506CF0A4F373B0F%219341&authkey=ANnjK3Uq1FhuAe8) (~10MB)

Antes de empezar la práctica, se recomienda descargar las imágenes y echarlas un vistazo.

## Carga de los datos

In [1]:
import cv2
import os
import numpy as np
import keras
from keras.datasets import cifar10
import matplotlib.pyplot as plt
import glob

# Keras descarga y divide automáticamente el dataset CIFAR-10 en datos de entrenamiento y prueba
print("Descargando/Cargando el dataset CIFAR-10...")
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Verificamos las dimensiones de los datos para confirmar que todo está correcto
print("--- Datos cargados con éxito ---")
print(f"Imágenes de entrenamiento: {x_train.shape}")
print(f"Etiquetas de entrenamiento: {y_train.shape}")
print(f"Imágenes de prueba: {x_test.shape}")
print(f"Etiquetas de prueba: {y_test.shape}")

Descargando/Cargando el dataset CIFAR-10...
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 13s 0us/step
--- Datos cargados con éxito ---
Imágenes de entrenamiento: (50000, 32, 32, 3)
Etiquetas de entrenamiento: (50000, 1)
Imágenes de prueba: (10000, 32, 32, 3)
Etiquetas de prueba: (10000, 1)


In [2]:
# Vamos a standarizar todas las imágenes a tamaño 64x64
IMG_SIZE = 64

In [ ]:
def load_train_set(dirname, map_characters, verbose=True):
    """Esta función carga los datos de training en imágenes.

    Como las imágenes tienen tamaños distintas, utilizamos la librería opencv
    para hacer un resize y adaptarlas todas a tamaño IMG_SIZE x IMG_SIZE.

    Args:
        dirname: directorio completo del que leer los datos
        map_characters: variable de mapeo entre labels y personajes
        verbose: si es True, muestra información de las imágenes cargadas

    Returns:
        X, y: X es un array con todas las imágenes cargadas con tamaño
                IMG_SIZE x IMG_SIZE
              y es un array con las labels de correspondientes a cada imagen
    """
    X_train = []
    y_train = []
    for label, character in map_characters.items():
        files = os.listdir(os.path.join(dirname, character))
        images = [file for file in files if file.endswith("jpg")]
        if verbose:
          print("Leyendo {} imágenes encontradas de {}".format(len(images), character))
        for image_name in images:
            image = cv2.imread(os.path.join(dirname, character, image_name))
            X_train.append(cv2.resize(image,(IMG_SIZE, IMG_SIZE)))
            y_train.append(label)
    return np.array(X_train), np.array(y_train)

In [ ]:
def load_test_set(dirname, map_characters, verbose=True):
    """Esta función funciona de manera equivalente a la función load_train_set
    pero cargando los datos de test."""
    X_test = []
    y_test = []
    reverse_dict = {v: k for k, v in map_characters.items()}
    for filename in glob.glob(dirname + '/*.*'):
        char_name = "_".join(filename.split('/')[-1].split('_')[:-1])
        if char_name in reverse_dict:
            image = cv2.imread(filename)
            image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
            X_test.append(image)
            y_test.append(reverse_dict[char_name])
    if verbose:
        print("Leídas {} imágenes de test".format(len(X_test)))
    return np.array(X_test), np.array(y_test)


In [ ]:
# Cargamos los datos. Si no estás trabajando en colab, cambia los paths por
# los de los ficheros donde hayas descargado los datos.
DATASET_TRAIN_PATH_COLAB = "/root/.keras/datasets/simpsons_train.tar.gz"
DATASET_TEST_PATH_COLAB = "/root/.keras/datasets/simpsons_test.tar.gz"

x_train, y_train = load_train_set(DATASET_TRAIN_PATH_COLAB, MAP_CHARACTERS)
x_test, y_test = load_test_set(DATASET_TEST_PATH_COLAB, MAP_CHARACTERS)

NotADirectoryError: [Errno 20] Not a directory: '/root/.keras/datasets/simpsons_train.tar.gz/abraham_grampa_simpson'

In [ ]:
# Vamos a barajar aleatoriamente los datos. Esto es importante ya que si no
# lo hacemos y, por ejemplo, cogemos el 20% de los datos finales como validation
# set, estaremos utilizando solo un pequeño número de personajes, ya que
# las imágenes se leen secuencialmente personaje a personaje.
perm = np.random.permutation(len(x_train))
x_train, y_train = x_train[perm], y_train[perm]

## Herramientas de visualización de resultados

In [ ]:
# Definición de funciones que permitirán la visualización de las graficas de entrenamiento
def plot_acc(history, title="Model Accuracy"):
    """Imprime una gráfica mostrando la accuracy por epoch obtenida en un entrenamiento"""
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title(title)
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(['Entrenamiento', 'Validación'], loc='upper left')
    plt.show()

def plot_loss(history, title="Model Loss"):
    """Imprime una gráfica mostrando la pérdida por epoch obtenida en un entrenamiento"""
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title(title)
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Entrenamiento', 'Validación'], loc='upper right')
    plt.show()

def plot_compare_losses(history1, history2, name1="Red 1",
                        name2="Red 2", title="Graph title"):
    """Compara losses de dos entrenamientos con nombres name1 y name2"""
    plt.plot(history1.history['loss'], color="green")
    plt.plot(history1.history['val_loss'], 'r--', color="green")
    plt.plot(history2.history['loss'], color="blue")
    plt.plot(history2.history['val_loss'], 'r--', color="blue")
    plt.title(title)
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Entrenamiento ' + name1, 'Validación ' + name1,
                'Entrenamiento ' + name2, 'Validación ' + name2],
               loc='upper right')
    plt.show()

def plot_compare_accs(history1, history2, name1="Red 1",
                      name2="Red 2", title="Graph title"):
    """Compara accuracies de dos entrenamientos con nombres name1 y name2"""
    plt.plot(history1.history['accuracy'], color="green")
    plt.plot(history1.history['val_accuracy'], 'r--', color="green")
    plt.plot(history2.history['accuracy'], color="blue")
    plt.plot(history2.history['val_accuracy'], 'r--', color="blue")
    plt.title(title)
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(['Train ' + name1, 'Val ' + name1,
                'Train ' + name2, 'Val ' + name2],
               loc='lower right')
    plt.show()



## Cosas a tener en cuenta:

A continuación se detallan una serie de aspectos orientativos que podrían ser analizados en vuestro informe

*   Realizar un análisis de los datos a utilizar.
* Recuerda partir los datos en training/validation para tener una buena estimación de los valores que nuestro modelo tendrá en los datos de test, así como comprobar que no estamos cayendo en overfitting. Una posible partición puede ser 80 / 20.
* Las imágenes **no están normalizadas**. Hay que normalizarlas como hemos hecho en trabajos anteriores.
* El test set del problema tiene imágenes un poco más "fáciles", por lo que es posible encontrarse con métricas en el test set bastante mejores que en el training set.
* Un error común en Keras es no instanciar un nuevo modelo cada vez que hacemos un nuevo entrenamiento. Al hacer

      *model = Sequential()*
      *model.add(lo que sea)  # Definición del modelo*
      *model.fit()*

    Si queremos entrenar un nuevo modelo o el mismo modelo otra vez, es necesario volver a inicializar el modelo con model = Sequential().
    Si olvidamos este paso y volvemos a hacer fit(), el modelo seguirá entrenando por donde se quedó en el último fit().
* Se recomienda construir una tabla con el mejor valor del acurracy y función de validación .
* Vamos a utilizar la misma arquitectura de red neuronal para todos los experimentos, que mostramos a continuación.

In [ ]:
from keras import layers
from keras import models
from keras.optimizers import Adamax, RMSprop, SGD
from keras.callbacks import EarlyStopping


# Definición y construcción del modelo 1
model = models.Sequential()
model.add(layers.Conv2D(64,(2,2), activation='relu', input_shape=(64, 64, 3), padding='same', name='Convolutiva-1'))
model.add(layers.MaxPooling2D(pool_size=(2,2), name='MaxPooling-1'))
model.add(layers.Flatten())
model.add(layers.Dense(128, activation='relu', name='Hidden-Layer-1'))
model.add(layers.Dense(64, activation='relu', name='Hidden-Layer-2'))
model.add(layers.Dense(18, activation='softmax', name='Output-Layer'))
model.summary()

## Realización de los experimentos

### Experimento 1: Visualización y preparación del dataset

  * Visualizar algunas imagenes aleatoriamente.
  * Comprobar el número de imagenes y formato.
  * Normalizar.
  * Cualquier otra acción que consideres oportuna que enriquezca el experimento.

Primeramente vamos a visualizar aleatoriamente algunas imagenes del dataset de training junto con su etiqueta.

In [ ]:
# --- Experimento 1: Visualización y preparación del dataset ---

# Información general del dataset
print(f'x_train shape: {x_train.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'x_test shape:  {x_test.shape}')
print(f'y_test shape:  {y_test.shape}')
print(f'Tipo de dato de las imágenes: {x_train.dtype}')
print(f'Rango de valores de píxeles: [{x_train.min()}, {x_train.max()}]')
print(f'Número de clases: {len(MAP_CHARACTERS)}')


In [ ]:
# Distribución de clases en entrenamiento
unique, counts = np.unique(y_train, return_counts=True)
plt.figure(figsize=(14, 5))
plt.bar([MAP_CHARACTERS[u].replace('_', ' ') for u in unique], counts, color='steelblue')
plt.xticks(rotation=90)
plt.title('Distribución de clases en el conjunto de entrenamiento')
plt.ylabel('Número de imágenes')
plt.tight_layout()
plt.show()


In [ ]:
# Visualizar 12 imágenes aleatorias con su etiqueta
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
random_indices = np.random.choice(len(x_train), 12, replace=False)
for i, ax in enumerate(axes.flat):
    idx_img = random_indices[i]
    # OpenCV lee en BGR, convertimos a RGB para visualizar
    img_rgb = cv2.cvtColor(x_train[idx_img], cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    ax.set_title(MAP_CHARACTERS[y_train[idx_img]].replace('_', ' '), fontsize=9)
    ax.axis('off')
plt.suptitle('Muestras aleatorias del dataset de entrenamiento', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Normalización de los datos (dividir entre 255 para obtener rango [0, 1])
x_train_norm = x_train.astype('float32') / 255.0
x_test_norm = x_test.astype('float32') / 255.0

print(f'Rango de valores después de normalizar: [{x_train_norm.min()}, {x_train_norm.max()}]')
print(f'Tipo de dato tras normalizar: {x_train_norm.dtype}')


**Conclusión Experimento 1:** El dataset contiene imágenes de 18 personajes de los Simpsons redimensionadas a 64×64 píxeles con 3 canales (BGR). Se observa un desbalanceo entre clases (algunos personajes tienen más imágenes que otros). Las imágenes se han normalizado al rango [0, 1] dividiendo entre 255, lo cual facilita la convergencia del entrenamiento.

### Experimento 2:  Relu vs Tangente hiperbólica

Para la realización de este experimento tiene que utilizar los siguientes hiperparámetros:


*   Optimizer: SGD
*   Loss: sparse_categorical_crossentropy
*   Metrics: accuracy
*   EarlyStopping
      *   monitor=val_loss
      *   patience = 2
      *   verbose=1
*   Batch_size: 32




Construcción del modelo Relu.

In [ ]:
# --- Experimento 2: ReLU vs Tanh ---

# Modelo con activación ReLU
model_relu = models.Sequential()
model_relu.add(layers.Conv2D(64, (2,2), activation='relu', input_shape=(64,64,3), padding='same', name='Conv-1'))
model_relu.add(layers.MaxPooling2D(pool_size=(2,2), name='MaxPool-1'))
model_relu.add(layers.Flatten())
model_relu.add(layers.Dense(128, activation='relu', name='Hidden-1'))
model_relu.add(layers.Dense(64, activation='relu', name='Hidden-2'))
model_relu.add(layers.Dense(18, activation='softmax', name='Output'))

model_relu.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_relu = model_relu.fit(x_train_norm, y_train, batch_size=32,
                              epochs=20, validation_split=0.2,
                              callbacks=[early_stop])
print('Mejor val_accuracy ReLU:', max(history_relu.history['val_accuracy']))


In [ ]:
# Modelo con activación Tanh
model_tanh = models.Sequential()
model_tanh.add(layers.Conv2D(64, (2,2), activation='tanh', input_shape=(64,64,3), padding='same', name='Conv-1'))
model_tanh.add(layers.MaxPooling2D(pool_size=(2,2), name='MaxPool-1'))
model_tanh.add(layers.Flatten())
model_tanh.add(layers.Dense(128, activation='tanh', name='Hidden-1'))
model_tanh.add(layers.Dense(64, activation='tanh', name='Hidden-2'))
model_tanh.add(layers.Dense(18, activation='softmax', name='Output'))

model_tanh.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_tanh = model_tanh.fit(x_train_norm, y_train, batch_size=32,
                              epochs=20, validation_split=0.2,
                              callbacks=[early_stop])
print('Mejor val_accuracy Tanh:', max(history_tanh.history['val_accuracy']))


In [ ]:
# Comparación ReLU vs Tanh
plot_compare_accs(history_relu, history_tanh, name1='ReLU', name2='Tanh', title='Accuracy: ReLU vs Tanh')
plot_compare_losses(history_relu, history_tanh, name1='ReLU', name2='Tanh', title='Loss: ReLU vs Tanh')

print(f'Mejor val_accuracy ReLU: {max(history_relu.history["val_accuracy"]):.4f}')
print(f'Mejor val_accuracy Tanh: {max(history_tanh.history["val_accuracy"]):.4f}')


**Conclusión Experimento 2:** Se comparan las funciones de activación ReLU y Tanh. Generalmente ReLU converge más rápido y evita el problema del gradiente desvaneciente, por lo que suele obtener mejor accuracy de validación. Nos quedamos con la función de activación que mejor val_accuracy obtenga para los siguientes experimentos.

### Experimento 3: Zero vs Glorot uniform



In [ ]:
# --- Experimento 3: Zeros vs Glorot Uniform ---

# Modelo con inicialización Zeros
model_zeros = models.Sequential()
model_zeros.add(layers.Conv2D(64, (2,2), activation='relu', input_shape=(64,64,3),
                              padding='same', kernel_initializer='zeros', name='Conv-1'))
model_zeros.add(layers.MaxPooling2D(pool_size=(2,2), name='MaxPool-1'))
model_zeros.add(layers.Flatten())
model_zeros.add(layers.Dense(128, activation='relu', kernel_initializer='zeros', name='Hidden-1'))
model_zeros.add(layers.Dense(64, activation='relu', kernel_initializer='zeros', name='Hidden-2'))
model_zeros.add(layers.Dense(18, activation='softmax', kernel_initializer='zeros', name='Output'))

model_zeros.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_zeros = model_zeros.fit(x_train_norm, y_train, batch_size=32,
                                epochs=20, validation_split=0.2,
                                callbacks=[early_stop])
print('Mejor val_accuracy Zeros:', max(history_zeros.history['val_accuracy']))


In [ ]:
# Modelo con inicialización Glorot Uniform (por defecto en Keras)
model_glorot = models.Sequential()
model_glorot.add(layers.Conv2D(64, (2,2), activation='relu', input_shape=(64,64,3),
                               padding='same', kernel_initializer='glorot_uniform', name='Conv-1'))
model_glorot.add(layers.MaxPooling2D(pool_size=(2,2), name='MaxPool-1'))
model_glorot.add(layers.Flatten())
model_glorot.add(layers.Dense(128, activation='relu', kernel_initializer='glorot_uniform', name='Hidden-1'))
model_glorot.add(layers.Dense(64, activation='relu', kernel_initializer='glorot_uniform', name='Hidden-2'))
model_glorot.add(layers.Dense(18, activation='softmax', kernel_initializer='glorot_uniform', name='Output'))

model_glorot.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_glorot = model_glorot.fit(x_train_norm, y_train, batch_size=32,
                                  epochs=20, validation_split=0.2,
                                  callbacks=[early_stop])
print('Mejor val_accuracy Glorot Uniform:', max(history_glorot.history['val_accuracy']))


In [ ]:
# Comparación Zeros vs Glorot Uniform
plot_compare_accs(history_zeros, history_glorot, name1='Zeros', name2='Glorot Uniform',
                  title='Accuracy: Zeros vs Glorot Uniform')
plot_compare_losses(history_zeros, history_glorot, name1='Zeros', name2='Glorot Uniform',
                    title='Loss: Zeros vs Glorot Uniform')

print(f'Mejor val_accuracy Zeros:         {max(history_zeros.history["val_accuracy"]):.4f}')
print(f'Mejor val_accuracy Glorot Uniform: {max(history_glorot.history["val_accuracy"]):.4f}')


**Conclusión Experimento 3:** Inicializar los pesos a cero provoca que todas las neuronas computen lo mismo (problema de simetría), impidiendo el aprendizaje. Glorot Uniform rompe esta simetría distribuyendo los pesos de forma adecuada, lo que permite una convergencia efectiva. Nos quedamos con Glorot Uniform.

### Experimento 4 - Aleatoria Normal vs Glorot uniform  


In [ ]:
# --- Experimento 4: Random Normal vs Glorot Uniform ---

# Modelo con inicialización Random Normal
model_rnorm = models.Sequential()
model_rnorm.add(layers.Conv2D(64, (2,2), activation='relu', input_shape=(64,64,3),
                              padding='same', kernel_initializer='random_normal', name='Conv-1'))
model_rnorm.add(layers.MaxPooling2D(pool_size=(2,2), name='MaxPool-1'))
model_rnorm.add(layers.Flatten())
model_rnorm.add(layers.Dense(128, activation='relu', kernel_initializer='random_normal', name='Hidden-1'))
model_rnorm.add(layers.Dense(64, activation='relu', kernel_initializer='random_normal', name='Hidden-2'))
model_rnorm.add(layers.Dense(18, activation='softmax', kernel_initializer='random_normal', name='Output'))

model_rnorm.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_rnorm = model_rnorm.fit(x_train_norm, y_train, batch_size=32,
                                epochs=20, validation_split=0.2,
                                callbacks=[early_stop])
print('Mejor val_accuracy Random Normal:', max(history_rnorm.history['val_accuracy']))


In [ ]:
# Modelo Glorot Uniform (reutilizamos la misma arquitectura)
model_glorot2 = models.Sequential()
model_glorot2.add(layers.Conv2D(64, (2,2), activation='relu', input_shape=(64,64,3),
                                padding='same', kernel_initializer='glorot_uniform', name='Conv-1'))
model_glorot2.add(layers.MaxPooling2D(pool_size=(2,2), name='MaxPool-1'))
model_glorot2.add(layers.Flatten())
model_glorot2.add(layers.Dense(128, activation='relu', kernel_initializer='glorot_uniform', name='Hidden-1'))
model_glorot2.add(layers.Dense(64, activation='relu', kernel_initializer='glorot_uniform', name='Hidden-2'))
model_glorot2.add(layers.Dense(18, activation='softmax', kernel_initializer='glorot_uniform', name='Output'))

model_glorot2.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_glorot2 = model_glorot2.fit(x_train_norm, y_train, batch_size=32,
                                    epochs=20, validation_split=0.2,
                                    callbacks=[early_stop])
print('Mejor val_accuracy Glorot Uniform:', max(history_glorot2.history['val_accuracy']))


In [ ]:
# Comparación Random Normal vs Glorot Uniform
plot_compare_accs(history_rnorm, history_glorot2, name1='Random Normal', name2='Glorot Uniform',
                  title='Accuracy: Random Normal vs Glorot Uniform')
plot_compare_losses(history_rnorm, history_glorot2, name1='Random Normal', name2='Glorot Uniform',
                    title='Loss: Random Normal vs Glorot Uniform')

print(f'Mejor val_accuracy Random Normal:  {max(history_rnorm.history["val_accuracy"]):.4f}')
print(f'Mejor val_accuracy Glorot Uniform: {max(history_glorot2.history["val_accuracy"]):.4f}')


**Conclusión Experimento 4:** Glorot Uniform está diseñado para mantener la varianza de las activaciones a lo largo de las capas, lo que suele dar lugar a un entrenamiento más estable y una mejor convergencia que una inicialización Random Normal genérica (media 0, std 0.05). Nos quedamos con Glorot Uniform.

### Experimento 5 - SGD vs RMSprop




In [ ]:
# --- Experimento 5: SGD vs RMSprop ---

# Modelo con SGD
model_sgd = models.Sequential()
model_sgd.add(layers.Conv2D(64, (2,2), activation='relu', input_shape=(64,64,3), padding='same', name='Conv-1'))
model_sgd.add(layers.MaxPooling2D(pool_size=(2,2), name='MaxPool-1'))
model_sgd.add(layers.Flatten())
model_sgd.add(layers.Dense(128, activation='relu', name='Hidden-1'))
model_sgd.add(layers.Dense(64, activation='relu', name='Hidden-2'))
model_sgd.add(layers.Dense(18, activation='softmax', name='Output'))

model_sgd.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_sgd = model_sgd.fit(x_train_norm, y_train, batch_size=32,
                            epochs=20, validation_split=0.2,
                            callbacks=[early_stop])
print('Mejor val_accuracy SGD:', max(history_sgd.history['val_accuracy']))


In [ ]:
# Modelo con RMSprop
model_rmsprop = models.Sequential()
model_rmsprop.add(layers.Conv2D(64, (2,2), activation='relu', input_shape=(64,64,3), padding='same', name='Conv-1'))
model_rmsprop.add(layers.MaxPooling2D(pool_size=(2,2), name='MaxPool-1'))
model_rmsprop.add(layers.Flatten())
model_rmsprop.add(layers.Dense(128, activation='relu', name='Hidden-1'))
model_rmsprop.add(layers.Dense(64, activation='relu', name='Hidden-2'))
model_rmsprop.add(layers.Dense(18, activation='softmax', name='Output'))

model_rmsprop.compile(optimizer=RMSprop(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_rmsprop = model_rmsprop.fit(x_train_norm, y_train, batch_size=32,
                                    epochs=20, validation_split=0.2,
                                    callbacks=[early_stop])
print('Mejor val_accuracy RMSprop:', max(history_rmsprop.history['val_accuracy']))


In [ ]:
# Comparación SGD vs RMSprop
plot_compare_accs(history_sgd, history_rmsprop, name1='SGD', name2='RMSprop',
                  title='Accuracy: SGD vs RMSprop')
plot_compare_losses(history_sgd, history_rmsprop, name1='SGD', name2='RMSprop',
                    title='Loss: SGD vs RMSprop')

print(f'Mejor val_accuracy SGD:     {max(history_sgd.history["val_accuracy"]):.4f}')
print(f'Mejor val_accuracy RMSprop: {max(history_rmsprop.history["val_accuracy"]):.4f}')


**Conclusión Experimento 5:** RMSprop adapta el learning rate por cada parámetro, lo que normalmente permite una convergencia más rápida y mejores resultados que SGD estándar. Se observa que RMSprop alcanza mejor accuracy de validación en menos epochs.

### Experimento 6: SGD vs Adamax

Probar con learning_rate=0.002, beta_1=0.9, beta_2=0.999

In [ ]:
# --- Experimento 6: SGD vs Adamax ---

# Modelo con SGD
model_sgd2 = models.Sequential()
model_sgd2.add(layers.Conv2D(64, (2,2), activation='relu', input_shape=(64,64,3), padding='same', name='Conv-1'))
model_sgd2.add(layers.MaxPooling2D(pool_size=(2,2), name='MaxPool-1'))
model_sgd2.add(layers.Flatten())
model_sgd2.add(layers.Dense(128, activation='relu', name='Hidden-1'))
model_sgd2.add(layers.Dense(64, activation='relu', name='Hidden-2'))
model_sgd2.add(layers.Dense(18, activation='softmax', name='Output'))

model_sgd2.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_sgd2 = model_sgd2.fit(x_train_norm, y_train, batch_size=32,
                              epochs=20, validation_split=0.2,
                              callbacks=[early_stop])
print('Mejor val_accuracy SGD:', max(history_sgd2.history['val_accuracy']))


In [ ]:
# Modelo con Adamax (con los hiperparámetros indicados)
model_adamax = models.Sequential()
model_adamax.add(layers.Conv2D(64, (2,2), activation='relu', input_shape=(64,64,3), padding='same', name='Conv-1'))
model_adamax.add(layers.MaxPooling2D(pool_size=(2,2), name='MaxPool-1'))
model_adamax.add(layers.Flatten())
model_adamax.add(layers.Dense(128, activation='relu', name='Hidden-1'))
model_adamax.add(layers.Dense(64, activation='relu', name='Hidden-2'))
model_adamax.add(layers.Dense(18, activation='softmax', name='Output'))

model_adamax.compile(optimizer=Adamax(learning_rate=0.002, beta_1=0.9, beta_2=0.999),
                     loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_adamax = model_adamax.fit(x_train_norm, y_train, batch_size=32,
                                  epochs=20, validation_split=0.2,
                                  callbacks=[early_stop])
print('Mejor val_accuracy Adamax:', max(history_adamax.history['val_accuracy']))


In [ ]:
# Comparación SGD vs Adamax
plot_compare_accs(history_sgd2, history_adamax, name1='SGD', name2='Adamax',
                  title='Accuracy: SGD vs Adamax')
plot_compare_losses(history_sgd2, history_adamax, name1='SGD', name2='Adamax',
                    title='Loss: SGD vs Adamax')

print(f'Mejor val_accuracy SGD:    {max(history_sgd2.history["val_accuracy"]):.4f}')
print(f'Mejor val_accuracy Adamax: {max(history_adamax.history["val_accuracy"]):.4f}')


**Conclusión Experimento 6:** Adamax es una variante de Adam basada en la norma infinita. Con los parámetros indicados (lr=0.002, beta_1=0.9, beta_2=0.999) suele superar a SGD ya que adapta el learning rate de forma individual por parámetro, logrando una convergencia más rápida y estable.

### Experimento 7: Aumento batch size 512

In [ ]:
# --- Experimento 7: Aumento de batch size a 512 ---
# Usamos el mejor optimizador encontrado hasta ahora.

# Modelo con batch_size=32 (baseline)
model_bs32 = models.Sequential()
model_bs32.add(layers.Conv2D(64, (2,2), activation='relu', input_shape=(64,64,3), padding='same', name='Conv-1'))
model_bs32.add(layers.MaxPooling2D(pool_size=(2,2), name='MaxPool-1'))
model_bs32.add(layers.Flatten())
model_bs32.add(layers.Dense(128, activation='relu', name='Hidden-1'))
model_bs32.add(layers.Dense(64, activation='relu', name='Hidden-2'))
model_bs32.add(layers.Dense(18, activation='softmax', name='Output'))

model_bs32.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_bs32 = model_bs32.fit(x_train_norm, y_train, batch_size=32,
                              epochs=20, validation_split=0.2,
                              callbacks=[early_stop])
print('Mejor val_accuracy batch_size=32:', max(history_bs32.history['val_accuracy']))


In [ ]:
# Modelo con batch_size=512
model_bs512 = models.Sequential()
model_bs512.add(layers.Conv2D(64, (2,2), activation='relu', input_shape=(64,64,3), padding='same', name='Conv-1'))
model_bs512.add(layers.MaxPooling2D(pool_size=(2,2), name='MaxPool-1'))
model_bs512.add(layers.Flatten())
model_bs512.add(layers.Dense(128, activation='relu', name='Hidden-1'))
model_bs512.add(layers.Dense(64, activation='relu', name='Hidden-2'))
model_bs512.add(layers.Dense(18, activation='softmax', name='Output'))

model_bs512.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_bs512 = model_bs512.fit(x_train_norm, y_train, batch_size=512,
                                epochs=20, validation_split=0.2,
                                callbacks=[early_stop])
print('Mejor val_accuracy batch_size=512:', max(history_bs512.history['val_accuracy']))


In [ ]:
# Comparación batch_size=32 vs batch_size=512
plot_compare_accs(history_bs32, history_bs512, name1='BS=32', name2='BS=512',
                  title='Accuracy: Batch Size 32 vs 512')
plot_compare_losses(history_bs32, history_bs512, name1='BS=32', name2='BS=512',
                    title='Loss: Batch Size 32 vs 512')

print(f'Mejor val_accuracy BS=32:  {max(history_bs32.history["val_accuracy"]):.4f}')
print(f'Mejor val_accuracy BS=512: {max(history_bs512.history["val_accuracy"]):.4f}')


**Conclusión Experimento 7:** Un batch size mayor (512) hace que cada actualización de pesos se base en más muestras, lo que produce gradientes más estables pero con menos actualizaciones por epoch. Con SGD, un batch size más grande suele producir peor generalización. Además, se requiere ajustar el learning rate para compensar. El batch_size=32 generalmente da mejores resultados.

### Experimento 8 - Aplicar BatchNormalization

In [ ]:
# --- Experimento 8: Aplicar BatchNormalization ---

# Modelo BASE sin BatchNormalization
model_no_bn = models.Sequential()
model_no_bn.add(layers.Conv2D(64, (2,2), activation='relu', input_shape=(64,64,3), padding='same', name='Conv-1'))
model_no_bn.add(layers.MaxPooling2D(pool_size=(2,2), name='MaxPool-1'))
model_no_bn.add(layers.Flatten())
model_no_bn.add(layers.Dense(128, activation='relu', name='Hidden-1'))
model_no_bn.add(layers.Dense(64, activation='relu', name='Hidden-2'))
model_no_bn.add(layers.Dense(18, activation='softmax', name='Output'))

model_no_bn.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_no_bn = model_no_bn.fit(x_train_norm, y_train, batch_size=32,
                                epochs=20, validation_split=0.2,
                                callbacks=[early_stop])
print('Mejor val_accuracy sin BN:', max(history_no_bn.history['val_accuracy']))


In [ ]:
# Modelo CON BatchNormalization
model_bn = models.Sequential()
model_bn.add(layers.Conv2D(64, (2,2), activation='relu', input_shape=(64,64,3), padding='same', name='Conv-1'))
model_bn.add(layers.BatchNormalization(name='BN-1'))
model_bn.add(layers.MaxPooling2D(pool_size=(2,2), name='MaxPool-1'))
model_bn.add(layers.Flatten())
model_bn.add(layers.Dense(128, activation='relu', name='Hidden-1'))
model_bn.add(layers.BatchNormalization(name='BN-2'))
model_bn.add(layers.Dense(64, activation='relu', name='Hidden-2'))
model_bn.add(layers.BatchNormalization(name='BN-3'))
model_bn.add(layers.Dense(18, activation='softmax', name='Output'))

model_bn.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_bn = model_bn.fit(x_train_norm, y_train, batch_size=32,
                          epochs=20, validation_split=0.2,
                          callbacks=[early_stop])
print('Mejor val_accuracy con BN:', max(history_bn.history['val_accuracy']))


In [ ]:
# Comparación sin BN vs con BN
plot_compare_accs(history_no_bn, history_bn, name1='Sin BN', name2='Con BN',
                  title='Accuracy: Sin BatchNorm vs Con BatchNorm')
plot_compare_losses(history_no_bn, history_bn, name1='Sin BN', name2='Con BN',
                    title='Loss: Sin BatchNorm vs Con BatchNorm')

print(f'Mejor val_accuracy sin BN: {max(history_no_bn.history["val_accuracy"]):.4f}')
print(f'Mejor val_accuracy con BN: {max(history_bn.history["val_accuracy"]):.4f}')


**Conclusión Experimento 8:** BatchNormalization normaliza las activaciones entre capas, lo que estabiliza y acelera el entrenamiento. Normalmente permite usar learning rates más altos y actúa como un ligero regularizador. Se observa una mejora en la convergencia y la accuracy de validación.

### Experimento 9 - Aumentar el número de parámentros por capa

Aumentar de la siguiente manera:

*   512 a la Conv2D
*   512 a la primera Dense
*   256 a la segunda Dense



In [ ]:
# --- Experimento 9: Aumentar el número de parámetros por capa ---
# Conv2D: 512 filtros, Dense1: 512, Dense2: 256

# Modelo base (64, 128, 64)
model_small = models.Sequential()
model_small.add(layers.Conv2D(64, (2,2), activation='relu', input_shape=(64,64,3), padding='same', name='Conv-1'))
model_small.add(layers.MaxPooling2D(pool_size=(2,2), name='MaxPool-1'))
model_small.add(layers.Flatten())
model_small.add(layers.Dense(128, activation='relu', name='Hidden-1'))
model_small.add(layers.Dense(64, activation='relu', name='Hidden-2'))
model_small.add(layers.Dense(18, activation='softmax', name='Output'))

model_small.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_small = model_small.fit(x_train_norm, y_train, batch_size=32,
                                epochs=20, validation_split=0.2,
                                callbacks=[early_stop])
print('Mejor val_accuracy modelo base:', max(history_small.history['val_accuracy']))


In [ ]:
# Modelo grande (512, 512, 256)
model_big = models.Sequential()
model_big.add(layers.Conv2D(512, (2,2), activation='relu', input_shape=(64,64,3), padding='same', name='Conv-1'))
model_big.add(layers.MaxPooling2D(pool_size=(2,2), name='MaxPool-1'))
model_big.add(layers.Flatten())
model_big.add(layers.Dense(512, activation='relu', name='Hidden-1'))
model_big.add(layers.Dense(256, activation='relu', name='Hidden-2'))
model_big.add(layers.Dense(18, activation='softmax', name='Output'))

model_big.summary()

model_big.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_big = model_big.fit(x_train_norm, y_train, batch_size=32,
                            epochs=20, validation_split=0.2,
                            callbacks=[early_stop])
print('Mejor val_accuracy modelo grande:', max(history_big.history['val_accuracy']))


In [ ]:
# Comparación modelo base vs modelo grande
plot_compare_accs(history_small, history_big, name1='Base (64,128,64)', name2='Grande (512,512,256)',
                  title='Accuracy: Base vs Modelo Grande')
plot_compare_losses(history_small, history_big, name1='Base (64,128,64)', name2='Grande (512,512,256)',
                    title='Loss: Base vs Modelo Grande')

print(f'Mejor val_accuracy Base:   {max(history_small.history["val_accuracy"]):.4f}')
print(f'Mejor val_accuracy Grande: {max(history_big.history["val_accuracy"]):.4f}')


**Conclusión Experimento 9:** Aumentar el número de parámetros incrementa la capacidad del modelo para aprender patrones más complejos. Sin embargo, también aumenta el riesgo de overfitting y el tiempo de entrenamiento. Si se observa la diferencia entre train y val accuracy creciendo, es señal de overfitting. El modelo más grande puede mejorar o no la val_accuracy dependiendo de la cantidad de datos disponibles.

### Experimento 10 - Aplicar Dropout 0.2

In [ ]:
# --- Experimento 10: Aplicar Dropout 0.2 ---

# Modelo sin Dropout (baseline)
model_no_drop = models.Sequential()
model_no_drop.add(layers.Conv2D(64, (2,2), activation='relu', input_shape=(64,64,3), padding='same', name='Conv-1'))
model_no_drop.add(layers.MaxPooling2D(pool_size=(2,2), name='MaxPool-1'))
model_no_drop.add(layers.Flatten())
model_no_drop.add(layers.Dense(128, activation='relu', name='Hidden-1'))
model_no_drop.add(layers.Dense(64, activation='relu', name='Hidden-2'))
model_no_drop.add(layers.Dense(18, activation='softmax', name='Output'))

model_no_drop.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_no_drop = model_no_drop.fit(x_train_norm, y_train, batch_size=32,
                                    epochs=20, validation_split=0.2,
                                    callbacks=[early_stop])
print('Mejor val_accuracy sin Dropout:', max(history_no_drop.history['val_accuracy']))


In [ ]:
# Modelo CON Dropout 0.2
model_drop = models.Sequential()
model_drop.add(layers.Conv2D(64, (2,2), activation='relu', input_shape=(64,64,3), padding='same', name='Conv-1'))
model_drop.add(layers.MaxPooling2D(pool_size=(2,2), name='MaxPool-1'))
model_drop.add(layers.Flatten())
model_drop.add(layers.Dense(128, activation='relu', name='Hidden-1'))
model_drop.add(layers.Dropout(0.2, name='Dropout-1'))
model_drop.add(layers.Dense(64, activation='relu', name='Hidden-2'))
model_drop.add(layers.Dropout(0.2, name='Dropout-2'))
model_drop.add(layers.Dense(18, activation='softmax', name='Output'))

model_drop.compile(optimizer=SGD(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

history_drop = model_drop.fit(x_train_norm, y_train, batch_size=32,
                              epochs=20, validation_split=0.2,
                              callbacks=[early_stop])
print('Mejor val_accuracy con Dropout:', max(history_drop.history['val_accuracy']))


In [ ]:
# Comparación sin Dropout vs con Dropout 0.2
plot_compare_accs(history_no_drop, history_drop, name1='Sin Dropout', name2='Con Dropout 0.2',
                  title='Accuracy: Sin Dropout vs Con Dropout 0.2')
plot_compare_losses(history_no_drop, history_drop, name1='Sin Dropout', name2='Con Dropout 0.2',
                    title='Loss: Sin Dropout vs Con Dropout 0.2')

print(f'Mejor val_accuracy sin Dropout: {max(history_no_drop.history["val_accuracy"]):.4f}')
print(f'Mejor val_accuracy con Dropout: {max(history_drop.history["val_accuracy"]):.4f}')


**Conclusión Experimento 10:** Dropout es una técnica de regularización que desactiva aleatoriamente un porcentaje de neuronas durante el entrenamiento. Esto reduce el overfitting al forzar a la red a no depender de neuronas específicas. Con Dropout=0.2 se espera una reducción en la brecha entre train y val accuracy, mejorando la capacidad de generalización del modelo.

###Anexos

Si os encontrais alguna anomalia mientras realizais el laboratorio, describirla en este punto, motivos del problema y solución.


In [ ]:
# --- Tabla resumen de todos los experimentos ---

print('=' * 70)
print(f'{"Experimento":<40} {"Mejor Val Accuracy":>20}')
print('=' * 70)

results = {
    'Exp2 - ReLU': max(history_relu.history['val_accuracy']),
    'Exp2 - Tanh': max(history_tanh.history['val_accuracy']),
    'Exp3 - Zeros': max(history_zeros.history['val_accuracy']),
    'Exp3 - Glorot Uniform': max(history_glorot.history['val_accuracy']),
    'Exp4 - Random Normal': max(history_rnorm.history['val_accuracy']),
    'Exp4 - Glorot Uniform': max(history_glorot2.history['val_accuracy']),
    'Exp5 - SGD': max(history_sgd.history['val_accuracy']),
    'Exp5 - RMSprop': max(history_rmsprop.history['val_accuracy']),
    'Exp6 - SGD': max(history_sgd2.history['val_accuracy']),
    'Exp6 - Adamax': max(history_adamax.history['val_accuracy']),
    'Exp7 - BS=32': max(history_bs32.history['val_accuracy']),
    'Exp7 - BS=512': max(history_bs512.history['val_accuracy']),
    'Exp8 - Sin BatchNorm': max(history_no_bn.history['val_accuracy']),
    'Exp8 - Con BatchNorm': max(history_bn.history['val_accuracy']),
    'Exp9 - Base (64,128,64)': max(history_small.history['val_accuracy']),
    'Exp9 - Grande (512,512,256)': max(history_big.history['val_accuracy']),
    'Exp10 - Sin Dropout': max(history_no_drop.history['val_accuracy']),
    'Exp10 - Con Dropout 0.2': max(history_drop.history['val_accuracy']),
}

for name, acc in results.items():
    print(f'{name:<40} {acc:>20.4f}')

print('=' * 70)
best = max(results, key=results.get)
print(f'\nMejor configuración: {best} con val_accuracy={results[best]:.4f}')


**Anomalías observadas:**

1. **Inicialización a ceros (Exp 3):** El modelo no aprende nada debido al problema de simetría. Todas las neuronas computan la misma salida y reciben el mismo gradiente, por lo que nunca se diferencian.

2. **EarlyStopping con patience=2:** En algunos experimentos, el EarlyStopping puede parar el entrenamiento demasiado pronto si hay fluctuaciones en la val_loss. Esto podría no dar tiempo suficiente a algunos optimizadores/configuraciones para converger.

3. **Batch size 512 (Exp 7):** Con un batch size tan grande y SGD sin ajustar el learning rate, el modelo puede tardar mucho más epochs en converger o directamente no aprender de forma efectiva.

4. **Dataset desbalanceado:** Algunos personajes tienen significativamente más imágenes que otros, lo que puede sesgar el modelo hacia las clases mayoritarias.